In [ ]:
!pip install transformers datasets accelerate torchvision


In [ ]:
'''from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation

feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b2-finetuned-ade-512-512")
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512",
    num_labels=10,
    ignore_mismatched_sizes=True
)
'''

In [ ]:
'''from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation

feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b5-finetuned-ade-640-640")
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b5-finetuned-ade-640-640",
    num_labels=10,
    ignore_mismatched_sizes=True
)'''

In [ ]:
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation

feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b4-finetuned-ade-512-512")
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b4-finetuned-ade-512-512",
    num_labels=10,
    ignore_mismatched_sizes=True
)

In [ ]:
'''from transformers import Mask2FormerImageProcessor, Mask2FormerForSemanticSegmentation

feature_extractor = Mask2FormerImageProcessor.from_pretrained("facebook/mask2former-swin-large-ade-semantic")
model = Mask2FormerForSemanticSegmentation.from_pretrained(
    "facebook/mask2former-swin-large-ade-semantic",
    ignore_mismatched_sizes=True,
    num_labels=10
)
'''

In [6]:
from torch.utils.data import Dataset
from PIL import Image
import os
import numpy as np
import torch

class VISPRDataset(Dataset):
    def __init__(self, image_dir, mask_dir, feature_extractor):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.feature_extractor = feature_extractor
        self.image_files = sorted(os.listdir(image_dir))  # assumes filenames align

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.image_files[idx].replace('.jpg', '.png'))

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path)

        inputs = self.feature_extractor(images=image, return_tensors="pt", size=(512, 512))
        mask = mask.resize((512, 512), resample=Image.NEAREST)
        inputs["labels"] = torch.tensor(np.array(mask), dtype=torch.long)


        # remove batch dimension
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs


In [16]:
train_dataset = VISPRDataset(
    image_dir="/kaggle/input/visual-redactions/images/train",
    mask_dir="/kaggle/input/visual-masks/masks/train",
    feature_extractor=feature_extractor
)

val_dataset = VISPRDataset(
    image_dir="/kaggle/input/visual-redactions/images/val",
    mask_dir="/kaggle/input/visual-masks/masks/val",
    feature_extractor=feature_extractor
)

test_dataset = VISPRDataset(
    image_dir="/kaggle/input/visual-redactions/images/test",
    mask_dir="/kaggle/input/visual-masks/masks/test",
    feature_extractor=feature_extractor
)


In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    save_total_limit=2,
    remove_unused_columns=False,  # very important for custom datasets
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)


In [9]:
trainer.train()


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,0.243768
2,0.680700,0.188213
3,0.211500,0.185972
4,0.151200,0.189194


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=1940, training_loss=0.298645880787643, metrics={'train_runtime': 6748.9079, 'train_samples_per_second': 2.295, 'train_steps_per_second': 0.287, 'total_flos': 4.678479538601066e+18, 'train_loss': 0.298645880787643, 'epoch': 4.0})

In [10]:
import shutil

# Zip the latest checkpoint
shutil.make_archive("segformer-b4-checkpoint-1940", 'zip', "results/checkpoint-1940")


'/kaggle/working/segformer-b4-checkpoint-1940.zip'

In [17]:
import torch
from torch.utils.data import DataLoader
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassAccuracy

# Reload best model
from transformers import SegformerForSemanticSegmentation
model = SegformerForSemanticSegmentation.from_pretrained("results/checkpoint-1940").to("cuda")
#Mean IoU: 0.33857202529907227
#Pixel Accuracy: 0.37947869300842285
# Setup dataloader
val_loader = DataLoader(test_dataset, batch_size=1)

# Setup metrics
NUM_CLASSES = 10
iou_metric = MulticlassJaccardIndex(num_classes=NUM_CLASSES).to("cuda")
acc_metric = MulticlassAccuracy(num_classes=NUM_CLASSES).to("cuda")


In [18]:
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassPrecision

iou_metric = MulticlassJaccardIndex(num_classes=10, average=None).to("cuda")
precision_metric = MulticlassPrecision(num_classes=10, average=None).to("cuda")


In [ ]:
model.eval()
l = len(val_loader)
b = 1
with torch.no_grad():
    for batch in val_loader:
        if b % 10 == 0:
            print(b,"/",l)
        b+=1
        pixel_values = batch["pixel_values"].to("cuda")
        labels = batch["labels"].to("cuda")

        outputs = model(pixel_values=pixel_values)
        logits = outputs.logits
        logits = torch.nn.functional.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
        preds = logits.argmax(dim=1)

        iou_metric.update(preds, labels)
        precision_metric.update(preds, labels)
        


In [20]:
# Your class names
class_names = [
    "background",
    "face",
    "license_plate",
    "person_body",
    "nudity",
    "handwriting",
    "disability",
    "medicine",
    "fingerprint",
    "signature"
]

# Compute
per_class_iou = iou_metric.compute().cpu().numpy()
per_class_precision = precision_metric.compute().cpu().numpy()

# Print nicely
print(f"{'Class':<15}  {'mIoU':>8}  {'Precision':>10}")
print("-" * 35)
for i, cls in enumerate(class_names):
    print(f"{cls:<15}  {per_class_iou[i]:8.3f}  {per_class_precision[i]:8.3f}")


Class                mIoU   Precision
-----------------------------------
background          0.946     0.974
face                0.771     0.871
license_plate       0.000     0.000
person_body         0.798     0.859
nudity              0.514     0.782
handwriting         0.684     0.806
disability          0.000     0.000
medicine            0.000     0.000
fingerprint         0.261     0.967
signature           0.000     0.000
